In [ ]:
# -*- coding: utf-8 -*-
# ==============================================================================
#  RAG + LLM LEGAL ASSISTANT  —  MULTI-MODEL BENCHMARK  (v3.1-aligned)
#  Doctoral Research Program · Predictive Legal-Strategy Model · Mexico (SCJN/LFT)
#  Corpus: SCJN resolutions (PDF) + Mexican labor-law PDFs · Domain: contratos
#  laborales (Ley Federal del Trabajo).
#
#  ── OBJECTIVE ─────────────────────────────────────────────────────────────────
#  Produce and evaluate a Spanish legal-recommendation assistant that generates a
#  litigation STRATEGY for labor-contract cases under Mexican law, and identify —
#  via a controlled benchmark — which base model best serves that objective inside
#  a FROZEN pure-RAG pipeline.
#
#  ── EXPERIMENTAL DESIGN (controlled two-factor comparison) ─────────────────────
#  The pipeline, prompts, generation budget and metrics are FROZEN across every
#  run; the ONLY thing that changes between runs is the GGUF loaded (set MODEL_KEY
#  in §2). Reasoning models are deliberately EXCLUDED to preserve the single-
#  variable design. The 5 runs decompose into two orthogonal, non-confounded axes:
#
#    Factor A — ARCHITECTURE @ fixed quantization (Q4_K_M), 7B–9B band:
#        llama31-8b-q4 · qwen25-7b-q4 · mistral-7b-v03-q4 · gemma2-9b-q4
#        → gaps attributable to architecture/scale alone.
#    Factor B — QUANTIZATION @ fixed architecture (Llama-3.1-8B), matched pair:
#        llama31-8b-q4 (Q4_K_M)  vs  llama31-8b-q8 (Q8_0)
#        → the only like-for-like quant contrast → reported as Δ_quant (§14).
#    llama31-8b-q4 anchors both axes AND shares the family of the GraphRAG arm,
#    bridging to the separate KG ablation (graph_rag_llm_legal_v3.1.py).
#
#  ── FROZEN PIPELINE INVARIANTS (identical across all 5 runs) ───────────────────
#    Retrieval : intfloat/multilingual-e5-large (L2-norm, cosine/IndexFlatIP);
#                ONE shared FAISS index; chunking prec 2000/200, leyes 2400/300; k=4.
#    Generation: byte-identical Spanish prompts; n_ctx=8192; trim budget 5500;
#                max_tokens 512/512/1024 (analysis/extraction/litigation);
#                deterministic decoding (T=0.1 analysis, T=0.0 extraction).
#    Task      : ternary label space {Procedente, Improcedente, Parcialmente
#                Improcedente} via normalise_decision; extract_case_to_json schema;
#                20-question rubric (LLM-as-judge).
#    Ablation  : KG_ENABLED=False (all 5 are pure-RAG arms), so they are comparable
#                to each other AND to the GraphRAG arm for the KG analysis.
#    NOTE: chat_format auto-detects per architecture — the *surface* control tokens
#    differ by design; the *semantic* prompt (instructions/context/query) is constant.
#
#  ── EVALUATION (full 90-case gold set → reported numbers; 5 cases → narrative) ──
#    Tier 1 (headline) : 3-class accuracy, macro-F1, MCC + per-class P/R/F1, each
#                        with 95% CASE-LEVEL BOOTSTRAP CIs; McNemar paired test vs
#                        the anchor (all models see the identical 90 cases).
#    Tier 2 (grounding): set-based article precision/recall/F1 vs gold articles.
#    Tier 3 (secondary): ROUGE-1/2/L, BLEU, BERTScore-F1 (fluency/adequacy proxy).
#    Strategy quality  : 20-question rubric score.
#    Reliability       : deterministic greedy decoding + fixed seed (re-runnable).
#
#  ── OUTPUTS ────────────────────────────────────────────────────────────────────
#    Per model : benchmark_results_<model>.csv, metrics_summary_<model>.csv (with
#                model/arch/quant metadata + CI bounds), confusion_matrix,
#                litigation_strategy, rubric_evaluation, qualitative_sample (§12b).
#    Comparison: §14 → benchmark_comparison.csv (metric × model, + Δ_quant/Δ_KG),
#                mcnemar_vs_base.csv, benchmark_comparison.png.
#
#  ── RUN PROTOCOL ───────────────────────────────────────────────────────────────
#    Per model: set MODEL_KEY (§2) → run top-to-bottom. §0 does the NumPy ABI reset
#    + CUDA llama-cpp install and restarts the kernel (Colab re-runs all cells), then
#    detects the sentinel and continues. §9 is GPU-gated and resumable (checkpoint).
#    After all 5 models: run §14 to emit the grouped comparison table.
# ==============================================================================

In [ ]:
# ──────────────────────────────────────────────────────────────────────────────
# §  Colab Drive mounting
# ──────────────────────────────────────────────────────────────────────────────
try:
    from google.colab import drive
    drive.mount("/content/drive", force_remount=False)
    _IN_COLAB = True
    print("  Google Drive mounted.")
except ImportError:
    _IN_COLAB = False

print("\n MyDrive mounted ...\n")
#!find /content/drive -iname '*qwen*.gguf' 2>/dev/null

!find /content/drive -iname '*.gguf' 2>/dev/null


In [ ]:
# ──────────────────────────────────────────────────────────────────────────────
# §0 · ENVIRONMENT SETUP  (v13 parity — replaces the legacy ABI-reset block)
#
# WHY (empirically established on the GraphRAG arm, same Colab image):
#   The legacy §0 uninstalled numpy/scipy/torchvision/torchaudio and force-pinned
#   the stack to torch 2.5.1+cu121 / numpy<2.0, then killed the kernel. That was
#   right for the ~2024 Colab image. It is now actively harmful: current Colab
#   ships torch 2.11.0+cu128 with torchvision/torchaudio matched and CUDA on, and
#   every scientific dependency imports cleanly. Forcing the old pins downgrades a
#   healthy environment into one that no longer matches the VM's CUDA driver,
#   which reproducibly caused numpy reverting to 2.5.x, torchvision vanishing, and
#   hard kernel deaths with NO Python traceback (ABI-mismatched C extensions
#   segfault — uncatchable). Verified: with §0 skipped on a clean VM, §1 failed
#   ONLY on genuinely absent packages (llama_cpp, langchain_community).
#   This cell installs what is missing and changes nothing that already works.
#
# llama-cpp-python wheel: upstream publishes cu121-cu125/cu13x but NOT cu128, so
#   we cannot index-match this VM exactly. cu121 works because CUDA minor-version
#   compatibility lets a 12.1 binary run on a 12.8 driver, and llama-cpp-python
#   links the SYSTEM CUDA runtime (it need not match torch's cu128).
#   --only-binary forces a LOUD failure rather than a silent CPU-only source
#   build, which would make the 5x60-case runs untenably slow.
# ──────────────────────────────────────────────────────────────────────────────
import subprocess, sys, importlib, os

print("§0  Environment setup (no ABI reset, no kernel restart)...")

_GPU = subprocess.run(["nvidia-smi"], capture_output=True).returncode == 0
# Downstream cells (§PRE, §3, §9) reference _HAS_GPU, exported by the legacy §0.
_HAS_GPU = _GPU
_GPU_NAME = ""
if _HAS_GPU:
    try:
        _GPU_NAME = subprocess.run(
            ["nvidia-smi", "--query-gpu=name", "--format=csv,noheader"],
            capture_output=True, text=True).stdout.strip().split("\n")[0]
    except Exception:
        _GPU_NAME = "GPU"
print(f"§0  GPU detected: {_GPU_NAME if _HAS_GPU else 'no — CPU runtime'}")

_PKGS = ["langchain", "langchain-community", "langchain-text-splitters",
         "sentence-transformers", "faiss-cpu", "pypdf", "PyPDF2",
         "evaluate", "rouge_score", "bert-score", "sacrebleu",
         "scikit-learn", "pandas", "matplotlib", "seaborn", "networkx", "spacy"]
print(f"§0  Installing {len(_PKGS)} pipeline packages (~2-3 min)...")
_r = subprocess.run([sys.executable, "-m", "pip", "install", "-q", *_PKGS],
                    capture_output=True, text=True)
print("  ✅ pipeline packages installed" if _r.returncode == 0
      else f"  ⚠️  pip returned {_r.returncode}:\n{_r.stdout[-1200:]}\n{_r.stderr[-1200:]}")

try:
    import llama_cpp
    print(f"  ✅ llama_cpp already present ({llama_cpp.__version__})")
except ModuleNotFoundError:
    print("§0  Installing llama-cpp-python (prebuilt CUDA wheel)...")
    if _HAS_GPU:
        _w = subprocess.run(
            [sys.executable, "-m", "pip", "install", "llama-cpp-python",
             "--prefer-binary", "--only-binary=llama-cpp-python",
             "--extra-index-url",
             "https://abetlen.github.io/llama-cpp-python/whl/cu121"],
            capture_output=True, text=True)
        if _w.returncode != 0:
            print("  ⚠️  cu121 wheel unavailable → retrying cu125...")
            _w = subprocess.run(
                [sys.executable, "-m", "pip", "install", "llama-cpp-python",
                 "--prefer-binary", "--only-binary=llama-cpp-python",
                 "--extra-index-url",
                 "https://abetlen.github.io/llama-cpp-python/whl/cu125"],
                capture_output=True, text=True)
        print("  ✅ llama-cpp-python installed" if _w.returncode == 0
              else f"  ❌ FAILED — do NOT proceed:\n{_w.stdout[-1200:]}\n{_w.stderr[-1200:]}")
    else:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                        "llama-cpp-python"], capture_output=True)

# spaCy model: this arm sets KG_ENABLED=False, but §1 imports spacy and the
# helper layer is shared with the GraphRAG arm, so the model is kept available.
_sp = subprocess.run([sys.executable, "-c",
                      "import spacy; spacy.load('es_core_news_lg'); print('ok')"],
                     capture_output=True, text=True)
if "ok" not in _sp.stdout:
    print("§0  Downloading spaCy es_core_news_lg (~600 MB)...")
    subprocess.run([sys.executable, "-m", "spacy", "download",
                    "es_core_news_lg", "--quiet"], capture_output=True)

print("\n§0  Verifying imports required by §1:")
_CHECK = ["torch", "torchvision", "torchaudio", "numpy", "pandas", "matplotlib",
          "seaborn", "networkx", "spacy", "llama_cpp", "langchain_community",
          "langchain_text_splitters", "evaluate", "bert_score", "sklearn"]
_missing = []
for _m in _CHECK:
    try:
        importlib.import_module(_m)
    except Exception as _e:
        _missing.append((_m, type(_e).__name__))
if _missing:
    print(f"  ❌ still missing/broken: {_missing}")
    print("  Do not run §1 yet — report this list.")
else:
    print(f"  ✅ all {len(_CHECK)} modules import cleanly — §1 is safe to run.")


In [ ]:
# ==============================================================================
#  §1 · Imports
# ==============================================================================

print("§1  Importing packages...")
import os, json, re, io, warnings, time as _time
warnings.filterwarnings("ignore")

import torch
# ── torch<2.7 compat shim (must precede any HuggingFace import) ───────────────
# Newer transformers/accelerate/peft reference float8 dtypes absent in torch 2.5.1
# (the v3.1-pinned version) without a hasattr guard. Alias to float32 so import-time
# lookups succeed; this pipeline never computes in fp8.
for _dt in ("float8_e8m0fnu", "float8_e4m3fn", "float8_e5m2",
            "float8_e4m3fnuz", "float8_e5m2fnuz"):
    if not hasattr(torch, _dt):
        setattr(torch, _dt, torch.float32)
print(f"  torch {torch.__version__} | CUDA: {torch.cuda.is_available()}")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import networkx as nx                      # kept for symmetry; KG stays empty in this arm

from llama_cpp import Llama
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS

import evaluate as hf_evaluate
from bert_score import score as bert_score_compute
from sklearn.metrics import (classification_report, confusion_matrix,
                             accuracy_score, f1_score)

try:
    from google.colab import drive
    drive.mount("/content/drive", force_remount=False)
    _IN_COLAB = True; print("  Google Drive mounted.\n")
except ImportError:
    _IN_COLAB = False
print("✅  Imports successful.\n")


In [ ]:
# ==============================================================================
#  §2 · Global constants  (invariants shared with v3.1 + arm flag)
# ==============================================================================

# ── Experimental arm identity ─────────────────────────────────────────────────
KG_ENABLED = False               # the SINGLE independent variable removed vs v3.1
# NOTE: ARM and ARM_SUFFIX are now DERIVED from the active model (see MODEL block
# below) so every model writes its own checkpoint/results and §14 can compare them.

DRIVE_ROOT      = "/content/drive/MyDrive/SCJN_RAG"
RAW_PDFS_PATH   = os.path.join(DRIVE_ROOT, "data/scjn_resoluciones")
LAW_PDFS_PATH   = os.path.join(DRIVE_ROOT, "data/leyes_mexicanas")
EVAL_JSON_PATH  = os.path.join(DRIVE_ROOT, "data/eval_json")
FAISS_INDEX_PATH= os.path.join(DRIVE_ROOT, "faiss_index")          # SAME index as v3.1
RUBRIC_JSON_PATH= os.path.join(DRIVE_ROOT, "data/eval_json",
                               "rubrica_validacion_contratos.json")
LOCAL_RAW_PDFS  = "/content/raw_pdfs"
LOCAL_LAW_PDFS  = "/content/law_pdfs"
LOCAL_JSON      = "/content/test_cases_json"

# ── FIX 2: persist the HuggingFace embedding model across sessions ────────────
# multilingual-e5-large (~2.2 GB) re-downloads on EVERY fresh runtime; at the
# 144 kB/s you hit, that is hours per run × 5 runs. Caching it to Drive makes runs
# 2–5 load it instantly. NOTE: HF_HOME cannot be used here — it is read when
# huggingface_hub is imported (§1), i.e. BEFORE this block runs — so §3.3 passes
# this path via `cache_folder=` instead, which works regardless of import order.
# Cost: ~2.2 GB of Drive quota. Set to None to fall back to ephemeral /root cache.
HF_CACHE_DIR = os.path.join(DRIVE_ROOT, "hf_cache")

# ══════════════════════════════════════════════════════════════════════════════
#  MODEL SLATE — controlled two-factor benchmark (one experiment = one MODEL_KEY)
# ──────────────────────────────────────────────────────────────────────────────
#  All entries are LOCAL GGUFs saved under Drive; the loader resolves each by GLOB
#  PATTERN across the candidate dirs, so exact filenames need not match — only that
#  exactly one .gguf matches. No reasoning models are included (single-variable
#  design): the pipeline, prompts, generation budget and metrics are FROZEN across
#  every run, so the ONLY thing that changes between runs is the GGUF loaded.
#
#  Two orthogonal factors are encoded in the metadata fields below:
#    • Factor A — ARCHITECTURE at fixed quantization (Q4_K_M):
#        llama31-8b-q4 · qwen25-7b-q4 · mistral-7b-v03-q4 · gemma2-9b-q4
#      → gaps here are attributable to architecture/scale (7B–9B band).
#    • Factor B — QUANTIZATION at fixed architecture (Llama-3.1-8B):
#        llama31-8b-q4  vs  llama31-8b-q8
#      → the only like-for-like quant contrast; reported with a Δ_quant column (§14).
#    llama31-8b-q4 sits at the intersection of both axes and anchors the design; it
#    is also the same family as the GraphRAG arm, bridging to the separate KG ablation.
#
#  Metadata fields per entry:
#    pattern     – glob to locate the GGUF on disk
#    arch        – architecture family (grouping key for §14 Factor A)
#    params_b    – parameter count in billions (size ladder)
#    quant       – quantization label (grouping key for §14 Factor B)
#    chat_format – None ⇒ auto-detect from the GGUF's embedded chat_template
#                  (correct per-architecture; surface control-tokens SHOULD differ,
#                   the semantic prompt is what stays constant). Override only if a
#                   model loads with a visibly wrong template.
#    reasoning   – all False here; kept so strip_reasoning() (§4) stays an inert,
#                  parity-safe no-op and the harness remains robust if extended.
# ══════════════════════════════════════════════════════════════════════════════
# FIX 1a: single definition (it was declared twice before — same value, but the
# duplicate invites drift). The shared-folder shortcut alternative kept as a note.
MODELS_DIR_DRIVE = os.path.join(DRIVE_ROOT, "models")
# MODELS_DIR_DRIVE = "/content/drive/.shortcut-targets-by-id/1du_8oLZStKNMxcW7eQV87ownGKkA8GnP/models"
MODELS_DIR_LOCAL = "/content/models"
MODEL_REGISTRY = {
  # ── Factor A: architecture @ Q4_K_M ─────────────────────────────────────────
  "llama31-8b-q4":     {"pattern": "*Llama-3.1-8B-Instruct-Q4_K_M*.gguf",
                        "arch": "Llama-3.1", "params_b": 8, "quant": "Q4_K_M",
                        "chat_format": None, "reasoning": False},
  "qwen25-7b-q4":      {"pattern": "*Qwen2.5-7B-Instruct*Q4_K_M*.gguf",
                        "arch": "Qwen2.5",   "params_b": 7, "quant": "Q4_K_M",
                        "chat_format": None, "reasoning": False},
  "mistral-7b-v03-q4": {"pattern": "*Mistral-7B-Instruct-v0.3*Q4_K_M*.gguf",
                        "arch": "Mistral-v0.3", "params_b": 7, "quant": "Q4_K_M",
                        "chat_format": None, "reasoning": False},
  "gemma2-9b-q4":      {"pattern": "*gemma-2-9b-it*Q4_K_M*.gguf",
                        "arch": "Gemma-2",   "params_b": 9, "quant": "Q4_K_M",
                        "chat_format": None, "reasoning": False,
                        "supports_system": False}, # ← Gemma has NO system role
  # ── Factor B: quantization @ Llama-3.1-8B (matched pair with llama31-8b-q4) ──
  "llama31-8b-q8":     {"pattern": "*Llama-3.1-8B-Instruct-Q8_0*.gguf",
                        "arch": "Llama-3.1", "params_b": 8, "quant": "Q8_0",
                        "chat_format": None, "reasoning": False,
                        # CORRECTIVE FIX: Q8_0 (~2x the VRAM of the Q4_K_M
                        # candidates) caused a CUDA OOM abort on this same
                        # model in the RL+LLM arm at the global N_CTX=8192.
                        # Trims the KV-cache budget for this candidate only;
                        # inference-engine setting, not part of the treatment.
                        "n_ctx": 6144},
}

# ►►► SELECT THE MODEL FOR THIS EXPERIMENT (run once per key; 5 runs total) ◄◄◄
MODEL_KEY = "llama31-8b-q4"
assert MODEL_KEY in MODEL_REGISTRY, f"MODEL_KEY '{MODEL_KEY}' not in {list(MODEL_REGISTRY)}"
MODEL_SPEC      = MODEL_REGISTRY[MODEL_KEY]
MODEL_ARCH      = MODEL_SPEC["arch"]
MODEL_PARAMS_B  = MODEL_SPEC["params_b"]
MODEL_QUANT     = MODEL_SPEC["quant"]
MODEL_REASONING = MODEL_SPEC.get("reasoning", False)
MODEL_SUPPORTS_SYSTEM = MODEL_SPEC.get("supports_system", True)   # default True
N_CTX           = MODEL_SPEC.get("n_ctx") or 8192  # per-model override or global
                                                    # (v3.1 invariant unless overridden above)

# ── JUDGE — fixed, independent, never a benchmark candidate ──────────────────
# CORRECTIVE FIX (post-hoc methodological review, RAG arm): this arm's rubric
# evaluation previously reused the CANDIDATE model as its own judge (self-
# grading), which introduces self-preference / self-enhancement bias in
# LLM-as-judge protocols (Zheng et al., 2023, "Judging LLM-as-a-Judge with
# MT-Bench and Chatbot Arena", NeurIPS) and broke comparability with the
# RL+LLM arm, which already used a fixed, out-of-family judge
# (Phi-3.5-mini-instruct). This block brings the arm into methodological
# alignment: ALL THREE arms (RAG+LLM, GraphRAG+LLM, RL+LLM) now score the
# 20-question rubric with the SAME fixed, independent judge — never the
# candidate under test.
JUDGE_KEY  = "phi35-mini-q4"
JUDGE_SPEC = {"pattern": "*Phi-3.5-mini-instruct*Q4_K_M*.gguf",
              "arch": "Phi-3.5", "params_b": 3.8, "quant": "Q4_K_M",
              "chat_format": None, "n_ctx": 4096, "supports_system": True,
              # Judge runs on CPU (see LazyLLM note in §3.1b): removes VRAM
              # contention with the candidate and holds the evaluation
              # instrument constant across all 15 arm x model conditions.
              "n_gpu_layers": 0}
assert MODEL_KEY != JUDGE_KEY, (
    "MODEL_KEY cannot equal JUDGE_KEY — the judge must stay out of the "
    "benchmark loop to avoid self-enhancement bias (Zheng et al., 2023).")

# ARM / artifact namespace derived from the model → no cross-experiment collisions.
ARM        = f"RAG_LLM_NO_KG__{MODEL_KEY}"
ARM_SUFFIX = f"ragonly_{MODEL_KEY}"

# Canonical TERNARY decision label space — identical to v3.1 (NOT the binary scheme).
DECISION_LABELS   = ["Procedente", "Improcedente", "Parcialmente Improcedente"]
DECISION_LABEL2ID = {v: i for i, v in enumerate(DECISION_LABELS)}

# Chunking & retrieval — identical to v3.1
CHUNK_SIZE_PREC, CHUNK_OVERLAP_PREC = 2000, 200
CHUNK_SIZE_LAW,  CHUNK_OVERLAP_LAW  = 2400, 300
RETRIEVER_K = 4

# ── PATCH A: GPU verification gate threshold ──────────────────────────────────
# CPU inference ≈0.8 tok/s; a T4 with offload ≈30–50 tok/s. 8 tok/s cleanly
# separates the two, so §9 aborts rather than silently running for hours on CPU.
MIN_TPS = 8.0

# ── PATCH B: checkpoint / resume ──────────────────────────────────────────────
RESUME           = True                       # set False to force a clean §9 run
CHECKPOINT_LOCAL = f"/content/checkpoint_{ARM_SUFFIX}.jsonl"
CHECKPOINT_DRIVE = os.path.join(DRIVE_ROOT, "results", ARM, f"checkpoint_{ARM_SUFFIX}.jsonl")
print(f"§2  Model={MODEL_KEY} | arch={MODEL_ARCH} {MODEL_PARAMS_B}B | quant={MODEL_QUANT} "
      f"| reasoning={MODEL_REASONING} | Arm={ARM} | KG_ENABLED={KG_ENABLED}")

print ("\n Modelos MODELS_DIR_DRIVE :\n", MODELS_DIR_DRIVE)
print ("\n Archivos MODELS_DIR_DRIVE :\n",
       os.listdir(MODELS_DIR_DRIVE) if os.path.isdir(MODELS_DIR_DRIVE) else "(dir not found)")

# ══════════════════════════════════════════════════════════════════════════
# CANONICAL ARTICLE NORMALIZER — single source of truth on Drive
# ══════════════════════════════════════════════════════════════════════════
# Legal article citations appear in incompatible surface forms across the gold
# set and the arms:
#     gold      "LFT arts. 20–21"   "CPEUM art. 123-A"
#     RAG/GRAG  "20 Ley Federal del Trabajo (LFT)"
#     RL        "LFT art. 12"
# Comparing these as raw strings (only .strip().lower()) scores a correct
# citation as a miss purely because of formatting. Measured impact on the
# 60-case draft: article F1 of 0.005 / 0.000 / 0.048 raw, versus 0.134 / 0.159
# / 0.207 canonicalized — i.e. the raw numbers were an artifact of output
# format, not of legal grounding.
#
# norm.py lives ONCE on Drive and is imported by ALL FOUR arms and by
# CrossModelCompar. A per-notebook copy would drift and produce two different
# article scores for the same run with no visible cause. Missing module is a
# HARD FAILURE — a silent fallback to raw matching is exactly the bug this
# module exists to remove.
_LIB_DIR = os.path.join(DRIVE_ROOT, "lib")
if _LIB_DIR not in sys.path:
    sys.path.insert(0, _LIB_DIR)
try:
    from norm import normalize_articles
except ModuleNotFoundError as _e:
    raise ModuleNotFoundError(
        f"Canonical article normalizer not found: {_LIB_DIR}/norm.py\n"
        f"  Upload norm.py to {_LIB_DIR}/ before running. Article grounding "
        f"metrics are invalid without it (raw string matching scores correct "
        f"citations as misses whenever the surface form differs from gold)."
    ) from _e
print(f"  ✅ canonical article normalizer loaded from {_LIB_DIR}/norm.py")

try:
    from index_manifest import ensure_manifest
except ModuleNotFoundError as _e:
    raise ModuleNotFoundError(
        f"index_manifest.py not found in {_LIB_DIR}/ — required to verify that "
        f"this arm retrieves from the same frozen index as the others."
    ) from _e


In [ ]:
# ==============================================================================
#  §3 · Model initialisation — LLM + embeddings
#  spaCy is not loaded in this arm (no KG). nlp = None.
# ==============================================================================

# ══════════════════════════════════════════════════════════════════════════════
#  §3.1 · Resolve the active GGUF — CASE-INSENSITIVE
#  Colab's ext4 filesystem is case-SENSITIVE, so a registry pattern written
#  "*Llama-3.1-8B-Instruct-Q4_K_M*.gguf" would NOT match a file named
#  "llama-3.1-8b-instruct-q4_k_m.gguf". We match with fnmatch against LOWERCASED
#  names: the human-readable pattern stays, but casing no longer breaks resolution.
#  Tokens/underscores/version digits must still be present (correct strictness —
#  prevents loading q8_0 when q4_k_m was intended).
# ══════════════════════════════════════════════════════════════════════════════
import shutil as _shutil_model, fnmatch as _fnmatch

def _resolve_gguf(pattern, search_dirs):
    """Case-insensitive, first-match GGUF resolver across an ordered dir list."""
    pat = pattern.lower()
    hits = []
    for d in search_dirs:
        if os.path.isdir(d):
            for fn in os.listdir(d):
                if fn.lower().endswith(".gguf") and _fnmatch.fnmatch(fn.lower(), pat):
                    hits.append(os.path.join(d, fn))
    return sorted(set(hits))

_SEARCH_DIRS = [MODELS_DIR_LOCAL, MODELS_DIR_DRIVE, "/content", DRIVE_ROOT]

print(f"§3.1  Loading model '{MODEL_KEY}'  (pattern: {MODEL_SPEC['pattern']})...")
_matches = _resolve_gguf(MODEL_SPEC["pattern"], _SEARCH_DIRS)
if not _matches:
    _avail = []
    for _d in _SEARCH_DIRS:
        if os.path.isdir(_d):
            _avail += [os.path.basename(p) for p in _resolve_gguf("*.gguf", [_d])]
    raise FileNotFoundError(
        f"No GGUF matching '{MODEL_SPEC['pattern']}' for MODEL_KEY='{MODEL_KEY}'.\n"
        f"  Searched: {_SEARCH_DIRS}\n"
        f"  .gguf files found: {sorted(set(_avail)) or '(none)'}\n"
        f"  → Fix the 'pattern' in MODEL_REGISTRY (§2), or place the file in the model "
        f"dir. Matching is case-insensitive, so casing is not the issue; check that the "
        f"model/quant tokens (e.g. 'q4_k_m') are present.")
if len(_matches) > 1:
    print(f"  ⚠️  {len(_matches)} files match; using the first: {os.path.basename(_matches[0])}")
    print(f"      (tighten the pattern in §2 to disambiguate — e.g. include the exact quant token.)")
_resolved = _matches[0]
print(f"  Resolved: {_resolved}")

# ── FIX 1: cache to local SSD (speed); never let caching crash the load ───────
# BUG BEING FIXED: the old test `_resolved.startswith("/content")` is TRUE for Drive
# paths too, because Drive mounts UNDER /content/drive/... . So a Drive-resident GGUF
# was mislabelled "local_ssd", the copy was skipped, and llama.cpp read the weights
# over Drive's FUSE layer → ~6 tok/s instead of 30–50 (and the §9 gate would abort).
# Local SSD now means: under /content but NOT under /content/drive.
_LOCAL_MODEL  = os.path.join("/content", os.path.basename(_resolved))
_on_local_ssd = (_resolved.startswith("/content")
                 and not _resolved.startswith("/content/drive"))
if _on_local_ssd:
    _model_path, _src = _resolved, "local_ssd"
elif os.path.exists(_LOCAL_MODEL):
    _model_path, _src = _LOCAL_MODEL, "local_ssd"
else:
    print("  Caching model to local SSD (faster than Drive-backed loading)...")
    try:
        os.makedirs(os.path.dirname(_LOCAL_MODEL) or ".", exist_ok=True)
        _shutil_model.copy2(_resolved, _LOCAL_MODEL)
        _model_path, _src = _LOCAL_MODEL, "drive_cache"
    except Exception as e:
        print(f"  ⚠️  Local cache failed ({e}) → loading directly from source.")
        _model_path, _src = _resolved, "drive_direct"

# ── Load with per-model chat_format (None ⇒ auto-detect from GGUF template) ────
_llama_kwargs = dict(model_path=_model_path, n_ctx=N_CTX, n_gpu_layers=-1, verbose=False)
if MODEL_SPEC.get("chat_format"):
    _llama_kwargs["chat_format"] = MODEL_SPEC["chat_format"]
try:
    llm = Llama(**_llama_kwargs)
except Exception as e:
    raise RuntimeError(
        f"Failed to load '{MODEL_KEY}' ({os.path.basename(_resolved)}): {e}\n"
        f"  If this is an 'unknown architecture' error, the installed llama-cpp-python "
        f"is too old — delete the §0 sentinel and re-run so §0 installs a newer wheel, "
        f"or set an explicit chat_format in MODEL_REGISTRY (§2).")
print(f"  Loaded from: {_model_path}")
print(f"  Source: {_src} | chat_format: {MODEL_SPEC.get('chat_format') or 'auto'} | n_ctx={N_CTX}")

# Speed probe (identical diagnostic to v3.1)
_t0 = _time.time()
_p  = llm.create_chat_completion(messages=[{"role":"user","content":"Di 'Hola'."}],
                                 max_tokens=5, temperature=0.0)
_rate = _p.get("usage",{}).get("completion_tokens",5) / max(_time.time()-_t0, 1e-6)
print(f"✅  Model ready (~{_rate:.0f} tok/s). Est. §9 (~60 cases): "
      f"~{60*(512/max(_rate,1))/60:.0f} min")

# ══════════════════════════════════════════════════════════════════════════
# §3.1b · FIXED JUDGE MODEL — Phi-3.5-mini-instruct (independent of candidate)
# CORRECTIVE FIX (2nd pass, post-hoc): the judge was originally loaded here
# EAGERLY and held resident for the ENTIRE run alongside the candidate. That
# is safe for the smaller Q4 candidates, but for llama31-8b-q8 (Q8_0, ~2x the
# VRAM of the Q4 models) permanent co-residency of candidate + judge +
# embeddings model caused the SAME class of CUDA OOM abort observed in the
# RL+LLM arm on this model. Following the pattern already validated there,
# the judge now loads ON DEMAND (only around §11b rubric scoring) and
# releases immediately after — never co-resident with the candidate during
# strategy generation, which is where peak VRAM pressure actually occurs.
# ══════════════════════════════════════════════════════════════════════════
class LazyLLM:
    """Loads the fixed judge GGUF on first use; releases on demand. Keeps the
    judge and candidate from being simultaneously resident for longer than
    the §11b scoring window actually requires."""
    def __init__(self, model_key, spec, search_dirs):
        self.model_key, self.spec, self.search_dirs = model_key, spec, search_dirs
        self._llm = None

    @property
    def is_loaded(self):
        return self._llm is not None

    def get(self):
        if self._llm is None:
            print(f"  Loading fixed judge '{self.model_key}' (on-demand)...")
            _matches = _resolve_gguf(self.spec["pattern"], self.search_dirs)
            if not _matches:
                raise FileNotFoundError(
                    f"No GGUF matching '{self.spec['pattern']}' for "
                    f"JUDGE_KEY='{self.model_key}'.\n  Searched: {self.search_dirs}\n"
                    f"  Place Phi-3.5-mini-instruct-Q4_K_M.gguf in {MODELS_DIR_DRIVE}/ "
                    f"before re-running — rubric scoring cannot proceed without a "
                    f"fixed, independent judge (see corrective methodology note in §2).")
            _resolved = _matches[0]
            _local = os.path.join("/content", os.path.basename(_resolved))
            if _resolved.startswith("/content") and not _resolved.startswith("/content/drive"):
                _path = _resolved
            elif os.path.exists(_local):
                _path = _local
            else:
                try:
                    _shutil_model.copy2(_resolved, _local); _path = _local
                except Exception as _e:
                    print(f"  ⚠️  Judge local cache failed ({_e}) → loading directly from source.")
                    _path = _resolved
            # FIX (v13 parity): read n_gpu_layers from the spec instead of
            # hardcoding -1. JUDGE_SPEC sets 0 (CPU) so the judge never competes
            # for VRAM with the candidate — required on a 16 GB T4 for the Q8
            # candidate, and kept constant across ALL conditions so the
            # evaluation instrument does not vary across the 3x5 matrix.
            _ngl = self.spec.get("n_gpu_layers", -1)
            _kwargs = dict(model_path=_path, n_ctx=self.spec["n_ctx"],
                           n_gpu_layers=_ngl, verbose=False)
            if self.spec.get("chat_format"):
                _kwargs["chat_format"] = self.spec["chat_format"]
            try:
                self._llm = Llama(**_kwargs)
            except Exception as _e:
                raise RuntimeError(
                    f"Failed to load fixed judge '{self.model_key}': {_e}\n"
                    f"  Rubric scoring cannot proceed without an independent judge. "
                    f"If VRAM is the issue, lower JUDGE_SPEC['n_ctx'] in §2.")
            print(f"  Judge loaded from: {_path} (n_ctx={self.spec['n_ctx']})")
        return self._llm

    def release(self):
        if self._llm is not None:
            del self._llm
            self._llm = None
            gc.collect()
            if torch.cuda.is_available():
                try:
                    torch.cuda.empty_cache()
                except Exception:
                    pass
            print(f"  Judge '{self.model_key}' released (VRAM freed).")

import gc
judge_mgr = LazyLLM(JUDGE_KEY, JUDGE_SPEC, _SEARCH_DIRS)
print(f"§3.1b  Fixed judge '{JUDGE_KEY}' configured for ON-DEMAND loading "
      f"(loads only around §11b, never held resident during generation).")



# ── §3.3 Embeddings — identical to v3.1 (multilingual-e5-large, normalized, cosine)
# FIX 2: cache_folder=HF_CACHE_DIR persists the ~2.2 GB download to Drive, so runs
# 2–5 load it instantly instead of re-downloading each session. Used instead of
# HF_HOME because HF_HOME is read at import time (§1), before §2 defines it.
_DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"§3.3  Loading multilingual-e5-large on device='{_DEVICE}'...")
if HF_CACHE_DIR:
    os.makedirs(HF_CACHE_DIR, exist_ok=True)
    print(f"  Embedding cache: {HF_CACHE_DIR}")
_emb_kwargs = dict(
    model_name="intfloat/multilingual-e5-large",
    model_kwargs={"device": _DEVICE},
    encode_kwargs={"normalize_embeddings": True, "batch_size": 64})
if HF_CACHE_DIR:
    _emb_kwargs["cache_folder"] = HF_CACHE_DIR
embeddings = HuggingFaceEmbeddings(**_emb_kwargs)

# Ablation: spaCy / KG disabled
nlp = None
legal_knowledge_graph = None
print(f"✅  Embeddings ready. spaCy/KG DISABLED (ablation, KG_ENABLED={KG_ENABLED}).\n")


In [ ]:
# ==============================================================================
#  §4 · Helper functions
#  Reproduced verbatim from v3.1 (same prompts, same JSON contract, same
#  metrics).
#  The two graph functions are retained for code symmetry but are never invoked
#  while KG_ENABLED is False; the graph slot in each prompt resolves to the empty
#  string.
# ==============================================================================

# ── 4.1  PDF loader ───────────────────────────────────────────────────────────
def load_pdfs_from_directory(directory_path: str) -> list:
    docs = []
    if not os.path.isdir(directory_path):
        print(f"  ⚠️  Directory not found: {directory_path}")
        return docs
    for fn in sorted(os.listdir(directory_path)):
        if fn.lower().endswith(".pdf"):
            try:
                docs.extend(PyPDFLoader(os.path.join(directory_path, fn)).load())
            except Exception as e:
                print(f"    ❌  {fn}: {e}")
    return docs

# ── 4.2  Gold benchmark loader (schema-identical to v3.1) ─────────────────────
def load_test_cases_gold(json_path: str) -> list:
    cases = []
    def _parse(data):
        if "cases" not in data: return []
        out = []
        for c in data["cases"]:
            gs  = c.get("gold_standard", {}); cel = c.get("expected_cel_output", {})
            bm  = c.get("benchmark_metadata", {})
            out.append({
                "case_id":       c.get("case_id", "UNKNOWN"),
                "query":         (c.get("input_summary","")+"\n\n"+c.get("facts","")).strip(),
                "ground_truth":  "\n".join(cel.get("conclusiones_legales", [])),
                "gold_decision": gs.get("decision", ""),
                "gold_articles": cel.get("articulos_citados", []),
                "confidence":    gs.get("confidence", ""),
                "difficulty":    bm.get("difficulty", ""),
                "category":      bm.get("category", ""),
                "critical":      bm.get("critical", False),
                "precedent_refs":gs.get("precedent_refs", []),
            })
        return out
    if os.path.isfile(json_path):
        with open(json_path, "r", encoding="utf-8") as f:
            cases = _parse(json.load(f))
        print(f"  Loaded {len(cases)} cases from '{json_path}'."); return cases
    if os.path.isdir(json_path):
        for jf in sorted(os.path.join(json_path,f) for f in os.listdir(json_path)
                         if f.lower().endswith(".json")):
            try:
                with open(jf,"r",encoding="utf-8") as f: cases.extend(_parse(json.load(f)))
            except Exception as e: print(f"    ❌  {jf}: {e}")
    print(f"  Loaded {len(cases)} cases total."); return cases

# ── 4.6/4.7  Graph helpers (retained for symmetry; unused while KG disabled) ───
def graph_aware_retriever(query, graph, nlp_model) -> dict:
    if graph is None or nlp_model is None:
        return {"nodes": [], "edges": []}
    doc = nlp_model(query)
    nodes, edges, sn, se = [], [], set(), set()
    for qe in [e.text for e in doc.ents]:
        for n in [n for n in graph.nodes if qe.lower() in n.lower()]:
            if n not in sn:
                nodes.append({"text": n, "label": graph.nodes[n].get("label","N/A")}); sn.add(n)
    return {"nodes": nodes, "edges": edges}

def format_graph_context_for_llm(query, ctx) -> str:
    if not ctx or (not ctx.get("nodes") and not ctx.get("edges")):
        return ""                          # ablation: empty graph context
    parts = [f"Consulta: {query}", "--- Grafo de Conocimiento ---"]
    parts += [f"  Entidad: {n['text']} (Tipo: {n['label']})" for n in ctx.get("nodes",[])]
    parts += [f"  Relación: ({e['head']}) -[{e['type']}]-> ({e['tail']})" for e in ctx.get("edges",[])]
    return "\n".join(parts)

# ── 4.8  Token-budget trim (verbatim from v3.1) ───────────────────────────────
def _trim_to_token_budget(text: str, model, max_tokens: int) -> str:
    try:
        tokens = model.tokenize(text.encode("utf-8"))
    except Exception:
        lim = int(max_tokens*2.8)
        return text[:lim]+" [...]" if len(text) > lim else text
    if len(tokens) <= max_tokens: return text
    try:
        t = model.detokenize(tokens[:max_tokens]).decode("utf-8", errors="ignore")
    except Exception:
        t = text[:int(max_tokens*2.8)]
    lp = t.rfind(".")
    if lp > len(t)//2: t = t[:lp+1]
    return t + " [...]"


# ── 4.8b  Reasoning-trace stripper (for DeepSeek-R1 / reasoning GGUFs) ─────────
def strip_reasoning(text: str) -> str:
    """Remove <think>…</think> chain-of-thought so the JSON extractor and the
    ROUGE/BLEU/BERTScore metrics see only the final answer. No-op for models that
    emit no such tags, so it is parity-safe across the whole model registry."""
    if not text:
        return text
    text = re.sub(r"<think>.*?</think>", "", text, flags=re.DOTALL | re.IGNORECASE)
    text = re.sub(r"</?think>", "", text, flags=re.IGNORECASE)   # unmatched tags
    return text.strip()


# ── 4.8c  Chat-message builder — per-architecture system-role transport ───────
# Gemma-2's chat template defines no system role; llama-cpp RAISES ("System role
# not supported") rather than folding it. Architectures differ in how a system
# instruction is TRANSPORTED, not in whether it is delivered. For models without a
# system role we prepend the identical system text to the user turn, so the SEMANTIC
# prompt (instructions + context + query) stays byte-identical across all 5 models —
# consistent with the documented invariant that surface control tokens may differ
# per architecture while the semantic prompt is frozen.
def _chat_messages(system: str, user: str) -> list:
    if MODEL_SUPPORTS_SYSTEM:
        return [{"role": "system", "content": system},
                {"role": "user",   "content": user}]
    return [{"role": "user", "content": f"{system}\n\n{user}"}]


# ── 4.9  RAG inference (v3.1 signature; KG branch ablated) ────────────────────
def perform_rag_legal_analysis(query, retriever, model,
                               knowledge_graph=None, nlp_model=None) -> tuple:
    _RAG_TOKEN_BUDGET = 5_500
    docs    = retriever.invoke(query)
    rag_ctx = _trim_to_token_budget("\n\n".join(d.page_content for d in docs),
                                    model, _RAG_TOKEN_BUDGET)

    # ── ABLATION: graph context is empty when KG disabled ─────────────────────
    if knowledge_graph is None:
        g_fmt = ""                               # <-- the single difference vs v3.1
    else:
        g_ctx = graph_aware_retriever(query, knowledge_graph, nlp_model)
        g_fmt = _trim_to_token_budget(
            format_graph_context_for_llm(query, g_ctx), model, 350)

    # Prompt template byte-identical to v3.1 (graph slot present, content removed)
    prompt = (f"CONTEXTO RAG:\n{rag_ctx}\n\n"
              f"CONTEXTO GRAFO:\n{g_fmt}\n\n"
              f"CONSULTA:\n{query}\n\nANÁLISIS LEGAL:")
    _sys = ("Eres un experto en derecho laboral mexicano. Responde con precisión "
            "técnica citando artículos de la LFT y jurisprudencia relevante.")

    full_text, token_count, _t = "", 0, _time.time()
    for ch in model.create_chat_completion(
            messages=_chat_messages(_sys, prompt),
#            messages=[{"role":"system","content":_sys},
#                      {"role":"user","content":prompt}],
            max_tokens=512, temperature=0.1, stream=True):
        d = ch["choices"][0]["delta"].get("content","")
        if d: full_text += d; token_count += 1
    print(f"\r    ✓  {token_count} tokens | "
          f"{token_count/max(_time.time()-_t,1e-6):.1f} tok/s     ", flush=True)
    analysis_text = strip_reasoning(full_text) if MODEL_REASONING else full_text
    return analysis_text, {"completion_tokens": token_count, "total_tokens": token_count}

# ── 4.9b  Structured JSON extractor (verbatim from v3.1; ternary label) ───────
def _clean_json_response(raw: str) -> str:
    raw = re.sub(r"<think>.*?</think>", "", raw, flags=re.DOTALL | re.IGNORECASE)
    raw = re.sub(r"^```(?:json)?\s*", "", raw, flags=re.MULTILINE)
    return re.sub(r"\s*```$", "", raw, flags=re.MULTILINE).strip()

def extract_case_to_json(analysis: str, model) -> dict:
    prompt = (f"Basado en el análisis legal, extrae la información en JSON:\n\n{analysis}\n\n"
              "Responde SOLO con este JSON. Para 'estado_del_procedimiento' usa ÚNICAMENTE "
              "una de estas tres opciones exactas: 'Procedente', 'Improcedente', "
              "'Parcialmente Improcedente'.\n"
              '{"partes_involucradas":[],"articulos_citados":[],'
              '"conclusiones_legales":[],"recomendaciones":[],'
              '"estado_del_procedimiento":"Procedente"}')
    resp = model.create_chat_completion(
        messages=_chat_messages("Solo respondes con JSON válido.", prompt),
        max_tokens=512, temperature=0.0)
#        messages=[{"role":"system","content":"Solo respondes con JSON válido."},
#                  {"role":"user","content":prompt}], max_tokens=512, temperature=0.0)
    try:
        return json.loads(_clean_json_response(resp["choices"][0]["message"]["content"]))
    except Exception:
        return {"error": "JSON parse failed"}

# ── 4.10  Metrics (verbatim from v3.1) ────────────────────────────────────────
def evaluate_text_generation(predictions: list, references: list) -> pd.DataFrame:
    rouge = hf_evaluate.load("rouge").compute(predictions=predictions, references=references)
    bleu  = hf_evaluate.load("bleu").compute(predictions=predictions,
                                             references=[[r] for r in references])
    try:
        _,_,F1 = bert_score_compute(predictions, references,
                                    model_type="bert-base-multilingual-cased",
                                    num_layers=9, lang="es", verbose=False)
        bert_f1 = round(float(F1.mean()), 4)
    except Exception as e:
        print(f"  ⚠️  BERTScore failed ({e})."); bert_f1 = 0.0
    return pd.DataFrame([{"ROUGE-1":round(rouge["rouge1"],4),
                          "ROUGE-2":round(rouge["rouge2"],4),
                          "ROUGE-L":round(rouge["rougeL"],4),
                          "BLEU":round(bleu["bleu"],4),
                          "BERTScore-F1":bert_f1}])

def _flatten_articles(arts) -> list:
    if not arts: return []
    out = []
    for it in arts:
        if isinstance(it, dict):
            out.append(" ".join(str(v) for v in it.values() if v is not None and str(v).strip()))
        elif isinstance(it, list):
            out.extend(str(x) for x in it if x is not None)
        elif it is not None:
            out.append(str(it))
    return out

def normalise_decision(raw: str) -> str:
    import difflib as _dl
    if not raw or not isinstance(raw, str): return ""
    low = raw.lower().strip()
    _proc = {"tramit","proceso","pendiente","análisis","revisión","estudio",
             "investigación","espera","curso","conocimiento","determinación",
             "resolución","diligencia"}
    if any(p in low for p in _proc): return ""
    if "parcialmente" in low: return "Parcialmente Improcedente"
    if "improcedente" in low or "no procede" in low or "no ha lugar" in low: return "Improcedente"
    if "procedente" in low or " procede" in low or "ha lugar" in low: return "Procedente"
    best = _dl.get_close_matches(raw, DECISION_LABELS, n=1, cutoff=0.5)
    return best[0] if best else ""

def compute_article_hit_at_k(pred_list: list, gold_list: list) -> dict:
    P,R,F = [],[],[]
    for pred, gold in zip(pred_list, gold_list):
    # CORRECTIVE FIX: `.strip().lower()` compares raw surface forms, so a
    # correct citation written differently from gold scored as a miss.
    # Measured on the 60-case draft: raw matching gave article F1 of 0.005
    # (RAG) / 0.000 (GraphRAG) / 0.048 (RL) — the ordering reflected output
    # FORMAT, not legal grounding. Canonicalized: 0.134 / 0.159 / 0.207.
    # Uses the same shared normalizer as CrossModelCompar so in-notebook and
    # benchmark-table numbers cannot diverge.
        ps = normalize_articles(_flatten_articles(pred))
        gs = normalize_articles(_flatten_articles(gold))
        if not gs: continue
        tp = len(ps & gs); p = tp/len(ps) if ps else 0.0; r = tp/len(gs)
        P.append(p); R.append(r); F.append(2*p*r/(p+r) if (p+r) else 0.0)
    return {"article_precision":round(np.mean(P),4) if P else 0.0,
            "article_recall":   round(np.mean(R),4) if R else 0.0,
            "article_F1":       round(np.mean(F),4) if F else 0.0}

def compute_classification_metrics(y_true, y_pred, label_names=None, suffix=ARM_SUFFIX):
    if label_names is None: label_names = DECISION_LABELS
    labels_ids = list(range(len(label_names)))
    print("\n  Classification Report:")
    print(classification_report(y_true, y_pred, labels=labels_ids,
                                target_names=label_names, zero_division=0))
    cm = confusion_matrix(y_true, y_pred, labels=labels_ids)
    fig, ax = plt.subplots(figsize=(7,5))
    sns.heatmap(cm, annot=True, fmt="d", cmap="Blues",
                xticklabels=label_names, yticklabels=label_names, ax=ax)
    ax.set_title(f"Confusion Matrix — Legal Decision ({ARM})")
    plt.tight_layout(); plt.savefig(f"/content/confusion_matrix_{suffix}.png", dpi=150); plt.show()

# ── 4.10b  Statistical layer: MCC + case-level bootstrap confidence intervals ──
# With the full 90-case gold set we can attach 95% CIs to the headline prediction
# metrics. macro-F1 and MCC have no clean closed-form CI, so we bootstrap over CASES
# (resample the paired (y_true, y_pred) vectors with replacement and recompute). This
# is the defensible per-model uncertainty estimate for the thesis.
from sklearn.metrics import matthews_corrcoef

def bootstrap_prediction_cis(y_true, y_pred, n_boot=2000, seed=42, alpha=0.05) -> dict:
    """Point estimate + (lo, hi) 95% percentile CI for accuracy, macro-F1, MCC."""
    y_true = np.asarray(y_true); y_pred = np.asarray(y_pred)
    n = len(y_true)
    out = {}
    if n == 0:
        return {m: {"point": float("nan"), "lo": float("nan"), "hi": float("nan")}
                for m in ("accuracy", "macro_f1", "mcc")}
    point = {"accuracy": accuracy_score(y_true, y_pred),
             "macro_f1": f1_score(y_true, y_pred, average="macro", zero_division=0),
             "mcc":      matthews_corrcoef(y_true, y_pred) if len(set(y_true)) > 1 else 0.0}
    rng = np.random.default_rng(seed)
    boot = {m: [] for m in point}
    for _ in range(n_boot):
        idx = rng.integers(0, n, n)                     # resample cases w/ replacement
        yt, yp = y_true[idx], y_pred[idx]
        boot["accuracy"].append(accuracy_score(yt, yp))
        boot["macro_f1"].append(f1_score(yt, yp, average="macro", zero_division=0))
        boot["mcc"].append(matthews_corrcoef(yt, yp) if len(set(yt)) > 1 else 0.0)
    for m in point:
        lo, hi = np.percentile(boot[m], [100*alpha/2, 100*(1-alpha/2)])
        out[m] = {"point": round(float(point[m]), 4),
                  "lo": round(float(lo), 4), "hi": round(float(hi), 4)}
    return out



# CORRECTIVE FIX (post-hoc review, RL+LLM arm): a candidate model can produce
# syntactically valid, non-empty JSON that nonetheless abandons the required
# schema entirely (e.g. wrapping content under an unexpected container key,
# or omitting the decision/grounding fields). json.loads() succeeding is not
# evidence of schema compliance — only of syntactic validity. A strategy in
# this state silently corrupts every downstream metric that reads a specific
# top-level key (grounding F1, decision accuracy), producing a spuriously low
# score that reflects a FORMAT violation rather than content quality. This
# validates the required keys are present before a parse is trusted; on
# failure the caller falls back to the same safe, correctly-keyed default
# dict already used for outright json.loads() failures.
_REQUIRED_STRATEGY_KEYS = ("fortalezas", "debilidades", "estrategia",
                           "articulos_clave", "probabilidad_exito")

def _is_valid_strategy_schema(d) -> bool:
    if not isinstance(d, dict):
        return False
    return all(k in d for k in _REQUIRED_STRATEGY_KEYS)


def litigation_strategy_analysis(facts, retriever, model,
                                 knowledge_graph=None, nlp_model=None) -> dict:
    docs    = retriever.invoke(facts)
    rag_ctx = _trim_to_token_budget("\n\n".join(d.page_content for d in docs), model, 4000)
    g_fmt = "" if knowledge_graph is None else _trim_to_token_budget(
        format_graph_context_for_llm(facts, graph_aware_retriever(facts, knowledge_graph, nlp_model)),
        model, 300)
    prompt = (f"CONTEXTO LEGAL (RAG + GRAFO DE CONOCIMIENTO):\n{rag_ctx}\n\n"
              f"ENTIDADES Y RELACIONES DEL GRAFO:\n{g_fmt}\n\n"
              f"HECHOS DEL CASO:\n{facts}\n\n"
              "Genera una estrategia de litigio ESTRUCTURADA en el siguiente formato JSON "
              "exacto. Para 'probabilidad_exito' usa ÚNICAMENTE: 'Alta', 'Media' o 'Baja'.\n\n"
              '{\n  "fortalezas": ["fortaleza legal 1", "fortaleza legal 2"],\n'
              '  "debilidades": ["debilidad o riesgo 1", "debilidad o riesgo 2"],\n'
              '  "estrategia": "descripción de la estrategia recomendada",\n'
              '  "articulos_clave": ["LFT art. 47", "LFT art. 48"],\n'
              '  "jurisprudencia_clave": ["tesis o jurisprudencia relevante"],\n'
              '  "probabilidad_exito": "Alta",\n'
              '  "razonamiento": "justificación jurídica detallada",\n'
              '  "advertencia_etica": "Esta predicción es orientativa y no sustituye '
              'el juicio de un abogado licenciado."\n}')
    raw = ""
    for ch in model.create_chat_completion(
            messages=_chat_messages(
                "Eres un abogado litigante experto en derecho laboral mexicano (LFT). "
                "Respondes ÚNICAMENTE con el JSON solicitado, sin texto adicional.",
                prompt), max_tokens=1024, temperature=0.1, stream=True):
        raw += ch["choices"][0]["delta"].get("content","")
#            messages=[{"role":"system","content":
#                       "Eres un abogado litigante experto en derecho laboral mexicano (LFT). "
#                       "Respondes ÚNICAMENTE con el JSON solicitado, sin texto adicional."},
#                      {"role":"user","content":prompt}],
#            max_tokens=1024, temperature=0.1, stream=True):
#        raw += ch["choices"][0]["delta"].get("content","")
    try:
        parsed = json.loads(_clean_json_response(raw))
        if not _is_valid_strategy_schema(parsed):
            print(f"      ⚠️  strategy discarded: valid JSON but non-compliant "
                  f"schema (missing {[k for k in _REQUIRED_STRATEGY_KEYS if k not in parsed]}) "
                  f"— using safe fallback instead.")
            raise ValueError("schema non-compliant")
    except Exception:
        parsed = {"fortalezas":[],"debilidades":[],"estrategia":raw,"articulos_clave":[],
                  "jurisprudencia_clave":[],"probabilidad_exito":"","razonamiento":raw,
                  "advertencia_etica":("Esta predicción es orientativa y no sustituye "
                                       "el juicio de un abogado licenciado.")}
    parsed["raw_text"] = raw
    return parsed

# ── Rubric (LLM-as-judge) — verbatim from v3.1 ────────────────────────────────
def load_rubric(json_path: str) -> list:
    for p in [json_path, os.path.join(LOCAL_JSON,"rubrica_validacion_contratos.json"),
              "/content/rubrica_validacion_contratos.json"]:
        if os.path.isfile(p):
            with open(p,"r",encoding="utf-8") as f: data = json.load(f)
            q = data.get("questions", [])
            print(f"  ✅  Rubric loaded: {len(q)} questions from '{p}'"); return q
    print("  ⚠️  Rubric JSON not found — rubric evaluation skipped."); return []

def evaluate_rubric(strategy: dict, rubric_questions: list, model) -> list:
    results = []
    s_txt = (f"Fortalezas: {'; '.join(strategy.get('fortalezas',[]))}\n"
             f"Debilidades: {'; '.join(strategy.get('debilidades',[]))}\n"
             f"Estrategia: {strategy.get('estrategia','')}\n"
             f"Artículos clave: {', '.join(strategy.get('articulos_clave',[]))}\n"
             f"Jurisprudencia: {', '.join(strategy.get('jurisprudencia_clave',[]))}\n"
             f"Probabilidad: {strategy.get('probabilidad_exito','')}\n"
             f"Razonamiento: {strategy.get('razonamiento','')}")
    for q in rubric_questions:
        crit = q["criteria"]
        crit_txt = "\n".join(f"  Criterio '{c['name']}' (peso {c['weight_pct']}%): {c['guideline']}"
                             for c in crit)
        jp = (f"Eres un abogado evaluador experto en derecho contractual mexicano.\n\n"
              f"PREGUNTA:\n{q['question']}\n\nRESPUESTA ESPERADA:\n{q['expected_answer']}\n\n"
              f"CRITERIOS:\n{crit_txt}\n\nESTRATEGIA A EVALUAR:\n{s_txt}\n\n"
              "Para cada criterio, determina si CUMPLE o NO CUMPLE. Responde ÚNICAMENTE con JSON:\n"
              + json.dumps({"question_id":q["id"],
                            "evaluations":[{"criterion":c["name"],"complies":True,
                                            "justification":"razón breve"} for c in crit]},
                           ensure_ascii=False))
        raw = ""
        try:
            for ch in model.create_chat_completion(
                    messages=_chat_messages(
                        "Eres un evaluador legal preciso. Respondes SÓLO con JSON válido.",
                        jp), max_tokens=512, temperature=0.0, stream=True):
                raw += ch["choices"][0]["delta"].get("content","")
#                    messages=[{"role":"system","content":
#                               "Eres un evaluador legal preciso. Respondes SÓLO con JSON válido."},
#                              {"role":"user","content":jp}],
#                    max_tokens=512, temperature=0.0, stream=True):
#                raw += ch["choices"][0]["delta"].get("content","")
            evals = json.loads(_clean_json_response(raw)).get("evaluations", [])
        except Exception:
            evals = [{"criterion":c["name"],"complies":False,"justification":"Error"} for c in crit]
        # CORRECTIVE FIX (post-hoc, RL+LLM arm's judge id-echo bug): matching by
        # exact string equality on the judge's ECHOED criterion name is fragile —
        # if the judge (Phi-3.5-mini) paraphrases, truncates, or re-accents the
        # label instead of reproducing it verbatim, the match silently fails and
        # a criterion the judge actually marked compliant gets scored 0. LLMs
        # preserve LIST ORDER far more reliably than they preserve exact string
        # labels, so we match evaluations to criteria POSITIONALLY instead —
        # the same fix already validated for the RL arm's grouped judge calls.
        cs = []
        for _pos, c in enumerate(crit):
            ev = evals[_pos] if _pos < len(evals) else None
            comp = bool(ev.get("complies",False)) if ev else False
            cs.append({"criterion":c["name"],"weight_pct":c["weight_pct"],"complies":comp,
                       "score":c["weight_pct"] if comp else 0.0,
                       "justification":ev.get("justification","") if ev else ""})
        qsc = sum(x["score"] for x in cs)
        results.append({"question_id":q["id"],"question":q["question"],
                        "question_score":round(qsc,2),"criteria":cs})
        st = "✅" if qsc>=66 else ("⚠️ " if qsc>=33 else "❌")
        print(f"    {st}  Q{q['id']:02d}: {qsc:.1f}% — {q['question'][:55]}...")
    return results

def compute_rubric_summary(rr: list) -> dict:
    if not rr: return {}
    sc = [r["question_score"] for r in rr]; fp = sum(sc)/len(sc)
    wp = ("Alta (≥80% cumplimiento)" if fp>=80 else
          "Media (60–79% cumplimiento)" if fp>=60 else "Baja (<60% cumplimiento)")
    return {"final_score_pct":round(fp,2),"win_probability":wp,
            "questions_passed":sum(1 for s in sc if s>=66),
            "questions_partial":sum(1 for s in sc if 33<=s<66),
            "questions_failed":sum(1 for s in sc if s<33),
            "question_scores":{f"Q{r['question_id']:02d}":r["question_score"] for r in rr}}

# ── PATCH A: inference-speed verification gate ────────────────────────────────
def verify_inference_speed(model, min_tps=MIN_TPS, hard=True) -> float:
    """Probe generation throughput. Aborts (hard=True) or warns (hard=False) if the
    model is running on CPU, so §9 never silently runs for hours."""
    _t = _time.time()
    r = model.create_chat_completion(
        messages=[{"role":"user","content":"Enumera los números del 1 al 25."}],
        max_tokens=64, temperature=0.0)
    n   = r.get("usage", {}).get("completion_tokens", 64)
    tps = n / max(_time.time() - _t, 1e-6)
    cuda = torch.cuda.is_available()
    print(f"  Speed probe: ~{tps:.1f} tok/s | torch.cuda={cuda}")
    if tps < min_tps:
        # FIX (v13 parity): the legacy sentinel/ABI-reset mechanism no longer
        # exists (§0 was replaced), so referencing _SENTINEL here raised a
        # NameError at the exact moment the user needed a readable diagnostic.
        if not cuda:
            diag = ("No GPU attached. Runtime → Change runtime type → T4 GPU, "
                    "then re-run from §0.")
        else:
            diag = ("GPU present but llama-cpp-python is a CPU build. Re-run §0; "
                    "if the cu121/cu125 wheel is unavailable for this VM, the "
                    "--only-binary flag will have failed loudly rather than "
                    "silently installing a CPU-only build.")
        banner = (f"\n{'='*70}\n  ❌  INFERENCE TOO SLOW: {tps:.1f} tok/s (< {min_tps}). "
                  f"§9 would take ~{60*(512/max(tps,0.1))/3600:.1f} h.\n  {diag}\n{'='*70}")
        if hard:
            raise RuntimeError(banner)
        print(banner)
    else:
        print(f"  ✅  GPU inference confirmed (~{tps:.1f} tok/s) → "
              f"§9 ETA ≈ {60*(512/tps)/60:.0f} min.")
    return tps

# ── PATCH B: checkpoint / resume helpers ──────────────────────────────────────
def _checkpoint_path() -> str:
    if RESUME and not os.path.exists(CHECKPOINT_LOCAL) and os.path.exists(CHECKPOINT_DRIVE):
        try:
            import shutil as _sh; _sh.copy2(CHECKPOINT_DRIVE, CHECKPOINT_LOCAL)
            print(f"  Restored checkpoint from Drive: {CHECKPOINT_DRIVE}")
        except Exception as e:
            print(f"  ⚠️  Could not restore Drive checkpoint: {e}")
    return CHECKPOINT_LOCAL

def load_checkpoint() -> list:
    p = _checkpoint_path()
    if not (RESUME and os.path.exists(p)): return []
    recs = []
    with open(p, "r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if line:
                try: recs.append(json.loads(line))
                except Exception: pass
    print(f"  ↻  Resuming: {len(recs)} cases already completed (from {p}).")
    return recs

def append_checkpoint(rec: dict) -> None:
    with open(CHECKPOINT_LOCAL, "a", encoding="utf-8") as f:
        f.write(json.dumps(rec, ensure_ascii=False) + "\n")
    try:                                       # best-effort mirror to Drive
        os.makedirs(os.path.dirname(CHECKPOINT_DRIVE), exist_ok=True)
        with open(CHECKPOINT_DRIVE, "a", encoding="utf-8") as f:
            f.write(json.dumps(rec, ensure_ascii=False) + "\n")
    except Exception:
        pass

print("✅  All helper functions defined.\n")


In [ ]:
# ==============================================================================
#  §5–§6 · PDF ingestion & FAISS index
#  Reuse the v3.1 FAISS index at FAISS_INDEX_PATH so both arms retrieve over an
#  identical vector store. The notebook rebuilds only if no cached index is
#  found.
# ==============================================================================

print("── §5  PDF Ingestion ──")
raw_src = RAW_PDFS_PATH if os.path.exists(RAW_PDFS_PATH) else LOCAL_RAW_PDFS
law_src = LAW_PDFS_PATH if os.path.exists(LAW_PDFS_PATH) else LOCAL_LAW_PDFS

_faiss_local = "/content/faiss_index"

# ── CORRECTIVE FIX (v13 parity) · FAISS/embedder dimensionality guard ────────
# A cached index built with a DIFFERENT embedding model loads WITHOUT error and
# only fails later, deep inside faiss.search (`assert d == self.d`), surfacing as
# a bare AssertionError on every case with no usable message.
# OBSERVED IN PRODUCTION (GraphRAG arm): a 384-dim index (MiniLM-family, 44,754
# vectors) persisted on Drive was silently reused against the 1024-dim
# intfloat/multilingual-e5-large embedder this pipeline specifies — all 90 cases
# failed, 0 checkpointed. The load path only rebuilt when NO cache existed and
# never validated what it loaded.
# METHODOLOGICAL NOTE: the index is shared across runs AND across arms, so the
# embedding model is part of the frozen retrieval substrate. A stale index means
# documented methodology (e5-large) and actual behaviour (MiniLM) diverge —
# every arm must retrieve over the SAME 1024-dim index for the ablation to
# isolate architecture rather than retrieval quality.
_FAISS_AUTO_REBUILD_ON_MISMATCH = True

def _assert_faiss_dim_matches(_db, _emb, _src_label):
    """Return True if the cached index width matches the embedder width."""
    _expected = len(_emb.embed_query("dim probe"))
    _actual = _db.index.d
    if _actual == _expected:
        print(f"  ✅  dim check OK — index.d={_actual} == embedder dim={_expected}")
        return True
    msg = (f"FAISS index loaded from {_src_label} has dim={_actual}, but the "
           f"configured embedder produces dim={_expected}. The cached index was "
           f"built with a DIFFERENT embedding model and CANNOT be used — every "
           f"retrieval would fail inside faiss.search with a bare AssertionError.")
    if not _FAISS_AUTO_REBUILD_ON_MISMATCH:
        raise RuntimeError(msg)
    print(f"  ⚠️  {msg}")
    import time as _t, shutil as _sh
    _stamp = _t.strftime("%Y%m%d_%H%M%S")
    for _p in [_faiss_local, FAISS_INDEX_PATH]:
        if os.path.exists(_p):
            _bak = f"{_p}__STALE_{_actual}dim_{_stamp}"
            try:
                _sh.move(_p, _bak)
                print(f"     moved aside (kept for audit): {_p} → {_bak}")
            except Exception as _e:
                print(f"     ⚠️  could not move {_p}: {_e}")
    return False

_faiss_ok = False
if os.path.exists(_faiss_local) or os.path.exists(FAISS_INDEX_PATH):
    _path = _faiss_local if os.path.exists(_faiss_local) else FAISS_INDEX_PATH
    print(f"  Loading cached FAISS index from {_path}...")
    vector_db = FAISS.load_local(_path, embeddings, allow_dangerous_deserialization=True)
    print(f"  ✅  {vector_db.index.ntotal:,} vectors.")
    _faiss_ok = _assert_faiss_dim_matches(vector_db, embeddings, _path)
    if _faiss_ok and _path != _faiss_local:
        vector_db.save_local(_faiss_local)

if not _faiss_ok:
    print("  No cached index — building (chunking identical to v3.1)...")
    all_docs = load_pdfs_from_directory(raw_src)
    law_docs = load_pdfs_from_directory(law_src)
    seps = ["\n\n","\n",".", " ", ""]
    texts_prec = RecursiveCharacterTextSplitter(
        chunk_size=CHUNK_SIZE_PREC, chunk_overlap=CHUNK_OVERLAP_PREC,
        separators=seps).split_documents(all_docs)
    texts_law = RecursiveCharacterTextSplitter(
        chunk_size=CHUNK_SIZE_LAW, chunk_overlap=CHUNK_OVERLAP_LAW,
        separators=seps).split_documents(law_docs)
    combined = texts_prec + texts_law
    print(f"  Chunks — prec:{len(texts_prec):,} law:{len(texts_law):,} total:{len(combined):,}")
    if not combined: raise RuntimeError("No chunks. Check PDF directory paths.")
    vector_db = FAISS.from_documents(combined, embeddings)
    vector_db.save_local(_faiss_local)
    try:
        vector_db.save_local(FAISS_INDEX_PATH)
    except Exception as e:
        print(f"  ⚠️  Drive save skipped: {e}")


# ── FIX 3 · frozen-index assertion (HARD-FAIL) ───────────────────────────────
# The dim guard above catches an embedder mismatch but NOT "same dim, different
# chunking" — two 1024-dim indexes built from different corpora look identical
# to it. The manifest pins embedder + dim + ntotal + chunk parameters, and any
# divergence stops the run. Auto-rebuild is deliberately NOT offered here:
# silent rebuild is the mechanism that let the arms drift apart originally.
ensure_manifest(FAISS_INDEX_PATH, vector_db, embedder="intfloat/multilingual-e5-large",
                chunk_prec=CHUNK_SIZE_PREC, overlap_prec=CHUNK_OVERLAP_PREC,
                chunk_law=CHUNK_SIZE_LAW,   overlap_law=CHUNK_OVERLAP_LAW,
                arm=ARM, embed_fn=embeddings.embed_query)

retriever = vector_db.as_retriever(search_kwargs={"k": RETRIEVER_K})
print(f"✅  FAISS RAG index ready ({vector_db.index.ntotal:,} vectors, k={RETRIEVER_K}).\n")

# §7 KG construction is INTENTIONALLY OMITTED in this arm (ablation).
print("── §7  Knowledge Graph: SKIPPED (KG_ENABLED=False, pure-RAG ablation) ──\n")


In [ ]:
# ==============================================================================
#  §8 · Benchmark & rubric loading
# ==============================================================================

print("── §8  Benchmark & Rubric Loading ──")
_eval_src  = EVAL_JSON_PATH if os.path.exists(EVAL_JSON_PATH) else LOCAL_JSON
_gold_file = os.path.join(_eval_src, "gold_prediction_90_compatible.json")
test_cases = load_test_cases_gold(_gold_file if os.path.isfile(_gold_file) else _eval_src)

if not test_cases:
    print("  ⚠️  No cases loaded. Using fallback demo case.")
    test_cases = [{"case_id":"DEMO-001",
        "query":("Trabajador demanda despido injustificado. El patrón no entregó aviso "
                 "escrito de rescisión. Tiene 5 años de antigüedad."),
        "ground_truth":("El despido es injustificado por ausencia de aviso escrito conforme "
                        "art. 47 LFT. Procede indemnización constitucional."),
        "gold_decision":"Procedente","gold_articles":["LFT art. 47","LFT art. 48","LFT art. 50"],
        "confidence":"high","difficulty":"básico","category":"Despido injustificado",
        "critical":True,"precedent_refs":[]}]

rubric_questions = load_rubric(RUBRIC_JSON_PATH)
print(f"✅  {len(test_cases)} benchmark cases | {len(rubric_questions)} rubric questions.\n")


In [ ]:
# ==============================================================================
#  §9 · Inference loop (RAG-only) — GPU-gated & resumable
#  knowledge_graph=None, nlp_model=None → KG context empty. Accumulators and
#  the
#  results_rows schema are identical to v3.1, plus an arm field for the join.
#  Patch A (GPU gate): the cell first probes throughput and aborts if < MIN_TPS
#  (default 8 tok/s) — CPU inference (~0.8 tok/s) would take hours, so it fails
#  fast with a
#  diagnosis instead of silently running. Patch B (resume): every completed
#  case is
#  appended to checkpoint_ragonly.jsonl (mirrored to Drive); on re-run the loop
#  rebuilds
#  the accumulators and processes only the remaining cases. A disconnect loses
#  at most the
#  in-flight case. Set RESUME=False (§2) or delete the checkpoint to force a
#  clean run.
# ==============================================================================

print("── §9  Inference Loop ──")

# ── PATCH A: hard GPU gate — abort here rather than run for hours on CPU ───────
verify_inference_speed(llm, min_tps=MIN_TPS, hard=True)

# ── PATCH B: resume from checkpoint — rebuild accumulators, skip finished cases ─
all_preds, all_refs, all_pred_arts, all_gold_arts = [], [], [], []
y_true_dec, y_pred_dec, results_rows, _loop_times = [], [], [], []
_done_ids = set()
for _rec in load_checkpoint():
    all_preds.append(_rec["analysis"]);       all_refs.append(_rec["ground_truth"])
    all_pred_arts.append(_rec["pred_arts"]);  all_gold_arts.append(_rec["gold_arts"])
    _pl = DECISION_LABEL2ID.get(_rec["pred_dec"], -1)
    _gl = DECISION_LABEL2ID.get(_rec["gold_dec"], -1)
    if _gl != -1 and _pl != -1: y_true_dec.append(_gl); y_pred_dec.append(_pl)
    results_rows.append(_rec["row"]); _done_ids.add(_rec["case_id"])
_n_done = len(_done_ids)

for i, case in enumerate(test_cases):
    cid, query = case.get("case_id", f"CASE-{i+1:03d}"), case.get("query","")
    gold_txt, gold_dec = case.get("ground_truth",""), case.get("gold_decision","")
    gold_arts = case.get("gold_articles", [])
    if cid in _done_ids:
        continue                                           # already completed (resume)
    if not query or not gold_txt:
        print(f"  ⚠️  Skipping {cid}: missing query/ground truth."); continue

    _remaining = len(test_cases) - i
    eta = (f" | ETA {_remaining*sum(_loop_times)/len(_loop_times)/60:.0f} min"
           if _loop_times else "")
    print(f"  [{i+1:02d}/{len(test_cases)}] {cid} "
          f"({case.get('difficulty','?')}/{case.get('category','?')}){eta}")
    _t = _time.time()
    try:
        analysis, usage = perform_rag_legal_analysis(
            query, retriever, llm, knowledge_graph=None, nlp_model=None)   # ABLATION
        structured = extract_case_to_json(analysis, llm)
        pred_arts  = _flatten_articles(structured.get("articulos_citados", []))
        pred_dec   = normalise_decision(structured.get("estado_del_procedimiento",""))
        pl, gl     = DECISION_LABEL2ID.get(pred_dec,-1), DECISION_LABEL2ID.get(gold_dec,-1)
        _sec = _time.time()-_t; _loop_times.append(_sec)
        row = {"arm":ARM,"model":MODEL_KEY,"case_id":cid,"category":case.get("category",""),
            "difficulty":case.get("difficulty",""),"critical":case.get("critical",False),
            "gold_decision":gold_dec,"pred_decision":pred_dec,
            "correct_decision":gold_dec==pred_dec,
            "n_gold_articles":len(gold_arts),"n_pred_articles":len(pred_arts),
            "prompt_tokens":usage.get("prompt_tokens",0),
            "completion_tokens":usage.get("completion_tokens",0),
            "inference_secs":round(_sec,1)}
        # in-memory accumulators
        all_preds.append(analysis); all_refs.append(gold_txt)
        all_pred_arts.append(pred_arts); all_gold_arts.append(gold_arts)
        if gl != -1 and pl != -1: y_true_dec.append(gl); y_pred_dec.append(pl)
        results_rows.append(row); _done_ids.add(cid)
        # PATCH B: persist immediately (a disconnect loses at most this one case)
        append_checkpoint({"case_id":cid,"analysis":analysis,"ground_truth":gold_txt,
                           "pred_arts":pred_arts,"gold_arts":gold_arts,
                           "pred_dec":pred_dec,"gold_dec":gold_dec,"row":row})
        print(f"         Gold: {gold_dec:35s} Pred: {pred_dec}  [{_sec:.0f}s]")
    except Exception as e:
        # CORRECTIVE FIX: str(e) can be EMPTY for some exception types (bare
        # AssertionError, certain low-level ABI/SystemError failures) —
        # printing only {e} then gives zero diagnostic signal (this is the
        # exact pattern that produced 60/60 identical blank failures (on the former 60-case set) under a
        # NumPy 2.x ABI mismatch). Print the type, repr, and traceback so the
        # ACTUAL cause is visible instead of guessed at.
        import traceback as _tb
        print(f"  ❌  {cid}: [{type(e).__name__}] {e!r}  (will retry on next run)")
        _tb.print_exc(limit=6)

print(f"\n✅  Inference loop complete. "
      f"{len(results_rows)} cases total ({_n_done} resumed, "
      f"{len(results_rows)-_n_done} new this run).")
print(f"   Checkpoint: {CHECKPOINT_LOCAL}")
print("   To force a clean re-run: set RESUME=False in §2, or delete the checkpoint file.\n")


In [ ]:
# ==============================================================================
#  §10 · Evaluation — same metric set as v3.1
# ==============================================================================

print("── §10  Evaluation ──")
valid = [(p,r) for p,r in zip(all_preds, all_refs) if p and r]
if valid:
    pl_, rl_ = zip(*valid)
    print("\n  Computing ROUGE / BLEU / BERTScore...")
    gen_df = evaluate_text_generation(list(pl_), list(rl_))
    print(gen_df.to_string(index=False))
else:
    gen_df = pd.DataFrame()

print("\n  Computing Article P/R/F1...")
art_metrics = compute_article_hit_at_k(all_pred_arts, all_gold_arts)
for k,v in art_metrics.items(): print(f"    {k}: {v}")

dec_acc = dec_macro_f1 = dec_mcc = float("nan")
dec_cis = {}
if y_true_dec and y_pred_dec:
    compute_classification_metrics(y_true_dec, y_pred_dec)
    dec_acc      = accuracy_score(y_true_dec, y_pred_dec)
    dec_macro_f1 = f1_score(y_true_dec, y_pred_dec, average="macro", zero_division=0)
    dec_mcc      = matthews_corrcoef(y_true_dec, y_pred_dec) if len(set(y_true_dec)) > 1 else 0.0
    # 95% case-level bootstrap CIs on the headline prediction metrics
    dec_cis = bootstrap_prediction_cis(y_true_dec, y_pred_dec)
    print(f"  Accuracy : {dec_acc:.4f}  95% CI [{dec_cis['accuracy']['lo']}, {dec_cis['accuracy']['hi']}]")
    print(f"  Macro F1 : {dec_macro_f1:.4f}  95% CI [{dec_cis['macro_f1']['lo']}, {dec_cis['macro_f1']['hi']}]")
    print(f"  MCC      : {dec_mcc:.4f}  95% CI [{dec_cis['mcc']['lo']}, {dec_cis['mcc']['hi']}]")

results_df = pd.DataFrame(results_rows)
if not results_df.empty:
    _csv = f"/content/benchmark_results_{ARM_SUFFIX}.csv"
    results_df.to_csv(_csv, index=False, encoding="utf-8")
    n_ok, n_tot = results_df["correct_decision"].sum(), len(results_df)
    print(f"\n  Decision accuracy : {n_ok}/{n_tot} ({100*n_ok/n_tot:.1f}%)")
    print(f"  Saved: {_csv}")


In [ ]:
# ==============================================================================
#  §11 · Litigation strategy — structured RAG-only output
# ==============================================================================

print("── §11  Litigation Strategy (RAG-only) ──")
_demo_facts = ("Un empleado con 5 años de antigüedad fue despedido verbalmente por su "
    "supervisor sin causa justificada. El patrón no entregó aviso escrito de rescisión, "
    "no invocó artículos de la LFT y no pagó partes proporcionales. El trabajador demanda "
    "indemnización constitucional, salarios caídos y prima de antigüedad conforme a la LFT.")
litigation_result = litigation_strategy_analysis(
    _demo_facts, retriever, llm, knowledge_graph=None, nlp_model=None)   # ABLATION

print(f"\n  Probabilidad de éxito : {litigation_result.get('probabilidad_exito','')}")
print(f"  Artículos clave       : {', '.join(litigation_result.get('articulos_clave',[]))}")
print("  Fortalezas:");  [print(f"    • {x}") for x in litigation_result.get("fortalezas",[])]
print("  Debilidades:"); [print(f"    • {x}") for x in litigation_result.get("debilidades",[])]
print(f"  Estrategia: {litigation_result.get('estrategia','')}")
print(f"  ⚠️  {litigation_result.get('advertencia_etica','')}")

_lit = f"/content/litigation_strategy_{ARM_SUFFIX}.json"
with open(_lit,"w",encoding="utf-8") as f:
    json.dump({"arm":ARM,"demo_facts":_demo_facts,
               "litigation_strategy":litigation_result}, f, indent=2, ensure_ascii=False)
print(f"  ✅  Saved: {_lit}")

# ==============================================================================
#  §11b · Rubric evaluation (20-question LLM-as-judge)
# ==============================================================================

print("── §11b  Rubric Evaluation ──")
rubric_results, rubric_summary = [], {}
if rubric_questions:
    _judge_llm = judge_mgr.get()  # loads on demand; not resident during generation
    rubric_results = evaluate_rubric(litigation_result, rubric_questions, _judge_llm)  # FIX: fixed judge, not candidate
    judge_mgr.release()
    rubric_summary = compute_rubric_summary(rubric_results)
    print(f"\n  Final score: {rubric_summary.get('final_score_pct',0):.1f}%  | "
          f"{rubric_summary.get('win_probability','')}")
    _rj = f"/content/rubric_evaluation_results_{ARM_SUFFIX}.json"
    with open(_rj,"w",encoding="utf-8") as f:
        json.dump({"arm":ARM,"model":MODEL_KEY,"judge_model":JUDGE_KEY,
                   "summary":rubric_summary,"results":rubric_results},
                  f, indent=2, ensure_ascii=False)
    print(f"  ✅  Saved: {_rj}")
else:
    print("  Rubric skipped (no questions loaded).")


In [ ]:
# ==============================================================================
#  §12 · Arm-tagged metrics summary & visualisation
#  metrics_summary_ragonly.csv carries (arm, metric, value) — the canonical
#  row-format that the GraphRAG arm and every future model also emit, so the
#  comparison is a simple concat.
# ==============================================================================

# ==============================================================================
#  §12 · Factor-tagged metrics summary & visualisation
#  metrics_summary_<model>.csv carries one row per metric with the factor metadata
#  (model / arch / params_b / quant / kg_enabled) attached, so §14 can concat all
#  models and GROUP the comparison table by metric × architecture × quantization.
#  Headline prediction metrics also emit *_ci_lo / *_ci_hi rows (95% bootstrap CI).
# ==============================================================================

print("── §12  Metrics Summary & Visualisation ──")
_META = {"arm": ARM, "model": MODEL_KEY, "arch": MODEL_ARCH,
         "params_b": MODEL_PARAMS_B, "quant": MODEL_QUANT, "kg_enabled": int(KG_ENABLED)}
def _mrow(metric, value):
    r = dict(_META); r["metric"] = metric; r["value"] = float(value); return r

_metrics_rows = []
# ── Tier 3 (secondary): generation fluency/adequacy ──────────────────────────
if not gen_df.empty:
    for col in gen_df.columns:
        _metrics_rows.append(_mrow(col, gen_df.iloc[0][col]))     # ROUGE-1/2/L, BLEU, BERTScore-F1
# ── Tier 2: citation grounding ───────────────────────────────────────────────
for k, v in art_metrics.items():
    _metrics_rows.append(_mrow(k, v))                              # article_precision/recall/F1
# ── Tier 1 (headline): prediction, with 95% bootstrap CIs ────────────────────
_metrics_rows.append(_mrow("decision_accuracy", dec_acc))
_metrics_rows.append(_mrow("decision_macro_f1", dec_macro_f1))
_metrics_rows.append(_mrow("decision_mcc",      dec_mcc))
if dec_cis:
    for _m, _key in [("accuracy","decision_accuracy"), ("macro_f1","decision_macro_f1"),
                     ("mcc","decision_mcc")]:
        _metrics_rows.append(_mrow(f"{_key}_ci_lo", dec_cis[_m]["lo"]))
        _metrics_rows.append(_mrow(f"{_key}_ci_hi", dec_cis[_m]["hi"]))
# ── Strategy quality (rubric, LLM-as-judge) ──────────────────────────────────
if rubric_summary:
    _metrics_rows.append(_mrow("rubric_final_score_pct", rubric_summary.get("final_score_pct", 0)))
    _metrics_rows.append(_mrow("rubric_questions_passed", rubric_summary.get("questions_passed", 0)))

metrics_summary_df = pd.DataFrame(_metrics_rows)
_msum = f"/content/metrics_summary_{ARM_SUFFIX}.csv"
metrics_summary_df.to_csv(_msum, index=False, encoding="utf-8")
print(metrics_summary_df[["metric","value"]].to_string(index=False)); print(f"  ✅  Saved: {_msum}")

# Decision distribution plot (same style as v3.1 §12.1)
if not results_df.empty:
    fig, axes = plt.subplots(1, 2, figsize=(13, 4))
    acc_by = results_df.groupby("difficulty")["correct_decision"].agg(["mean","count"]).reset_index()
    axes[0].bar(acc_by["difficulty"], acc_by["mean"], color="#9DC3E6", edgecolor="#1F3864")
    axes[0].set_ylim(0,1.05); axes[0].set_title(f"Decision Accuracy by Difficulty ({ARM})")
    for j,row in acc_by.iterrows():
        axes[0].text(j, row["mean"]+0.02, f"n={int(row['count'])}", ha="center", fontsize=9)
    gc = results_df["gold_decision"].value_counts(); pc = results_df["pred_decision"].value_counts()
    x = np.arange(len(DECISION_LABELS)); w = 0.35
    axes[1].bar(x-w/2, [gc.get(l,0) for l in DECISION_LABELS], w, label="Gold", color="#1F3864")
    axes[1].bar(x+w/2, [pc.get(l,0) for l in DECISION_LABELS], w, label="Predicted", color="#9DC3E6")
    axes[1].set_xticks(x); axes[1].set_xticklabels([l[:15] for l in DECISION_LABELS], rotation=10)
    axes[1].set_title("Decision Distribution"); axes[1].legend()
    plt.suptitle(f"RAG+LLM (no-KG) — Benchmark Results [{ARM}]", fontsize=13, fontweight="bold")
    plt.tight_layout(); plt.savefig(f"/content/benchmark_summary_plots_{ARM_SUFFIX}.png", dpi=150); plt.show()

print("\n✅  Pipeline complete. Artifacts:")
for a in [f"benchmark_results_{ARM_SUFFIX}.csv", f"confusion_matrix_{ARM_SUFFIX}.png",
          f"benchmark_summary_plots_{ARM_SUFFIX}.png", f"metrics_summary_{ARM_SUFFIX}.csv",
          f"litigation_strategy_{ARM_SUFFIX}.json", f"rubric_evaluation_results_{ARM_SUFFIX}.json"]:
    print(f"    /content/{a}")

# ==============================================================================
#  §12b · Qualitative sample — 5 representative cases for the thesis narrative
#  The full 90-case run drives the NUMBERS; here we select 5 diverse cases (spanning
#  difficulty × category) and dump their full records (facts → analysis → pred vs
#  gold) so the discussion chapter can show WHY a model wins/loses, not just that it
#  does. Reads the checkpoint (which retains the analysis text per case).
# ==============================================================================
print("── §12b  Qualitative 5-case sample ──")
_qual = []
try:
    _recs = load_checkpoint()
    _seen_cat, _picked = set(), []
    for _r in sorted(_recs, key=lambda r: (r["row"].get("difficulty",""),
                                           r["row"].get("category",""))):
        _cat = _r["row"].get("category","")
        if _cat not in _seen_cat:                 # prefer category diversity
            _seen_cat.add(_cat); _picked.append(_r)
        if len(_picked) >= 5: break
    if len(_picked) < 5:                          # top up if few categories
        _picked += [r for r in _recs if r not in _picked][:5-len(_picked)]
    for _r in _picked[:5]:
        _qual.append({"case_id": _r["case_id"], "category": _r["row"].get("category",""),
                      "difficulty": _r["row"].get("difficulty",""),
                      "gold_decision": _r["gold_dec"], "pred_decision": _r["pred_dec"],
                      "correct": _r["gold_dec"] == _r["pred_dec"],
                      "gold_articles": _r["gold_arts"], "pred_articles": _r["pred_arts"],
                      "analysis": _r["analysis"]})
    _qf = f"/content/qualitative_sample_{ARM_SUFFIX}.json"
    with open(_qf, "w", encoding="utf-8") as f:
        json.dump({"model": MODEL_KEY, "arch": MODEL_ARCH, "quant": MODEL_QUANT,
                   "cases": _qual}, f, indent=2, ensure_ascii=False)
    print(f"  ✅  {len(_qual)} representative cases → {_qf}")
except Exception as e:
    print(f"  ⚠️  Qualitative sample skipped: {e}")


In [ ]:
# ==============================================================================
#  §13 · Save artifacts to Google Drive
# ==============================================================================
print("── §13  Saving Artifacts to Drive ──")
import shutil as _shutil
_OUT = os.path.join(DRIVE_ROOT, "results", ARM)
os.makedirs(_OUT, exist_ok=True)
for fn in [f"benchmark_results_{ARM_SUFFIX}.csv", f"confusion_matrix_{ARM_SUFFIX}.png",
           f"benchmark_summary_plots_{ARM_SUFFIX}.png", f"metrics_summary_{ARM_SUFFIX}.csv",
           f"litigation_strategy_{ARM_SUFFIX}.json", f"rubric_evaluation_results_{ARM_SUFFIX}.json",
           f"qualitative_sample_{ARM_SUFFIX}.json"]:
    src = f"/content/{fn}"
    if os.path.exists(src):
        try:
            _shutil.copy2(src, os.path.join(_OUT, fn)); print(f"  ✅  {fn}")
        except Exception as e:
            print(f"  ⚠️  {fn}: {e}")
print(f"\n✅  Saved to {_OUT}")


In [ ]:
# ── PRE-FLIGHT: verify all 5 runs are complete and mutually comparable ────────
# Preventing the risk of that a model run silently completed fewer than the
# 90 cases (a CSV with 0 or >90 rows). If any model shows fewer, need to re-run
# that MODEL_KEY (the checkpoint resumes it) before proceeding.

import pandas as pd, glob, os
EXPECTED = ["llama31-8b-q4","qwen25-7b-q4","mistral-7b-v03-q4","gemma2-9b-q4","llama31-8b-q8"]

rows, case_sets = [], {}
for k in EXPECTED:
    hits = (glob.glob(os.path.join(DRIVE_ROOT,"results","**",f"benchmark_results_ragonly_{k}.csv"), recursive=True)
            + glob.glob(f"/content/benchmark_results_ragonly_{k}.csv"))
    if not hits:
        rows.append({"model":k,"cases":0,"status":"❌ MISSING — not run"}); continue
    d = pd.read_csv(sorted(hits)[-1])
    case_sets[k] = set(d["case_id"])
    rows.append({"model":k,"cases":len(d),
                 "acc":round(d["correct_decision"].mean(),4),
                 "status":"✅ ok" if len(d)==60 else f"⚠️  INCOMPLETE ({len(d)}/60)"})
print(pd.DataFrame(rows).to_string(index=False))

if len(case_sets) == len(EXPECTED):
    common = set.intersection(*case_sets.values())
    print(f"\nIdentical case sets across all 5? {all(s==common for s in case_sets.values())}"
          f" | common cases: {len(common)}")
    if len(common) < 60:
        print("⚠️  Models were NOT evaluated on identical cases — McNemar/comparison invalid.\n")
else:
    print(f"\n⚠️  Only {len(case_sets)}/5 models present — do not run §14 yet.\n")

In [ ]:
# ==============================================================================
#  §14 · Cross-model comparison  (run after all 5 models have been executed)
# ──────────────────────────────────────────────────────────────────────────────
# §14 reads from disk, so it doesn't need a model loaded. It can be executed in
# a fresh session by running §0 → §1 → §2 → §14 (skip §3's model load and
# §5–§13 entirely). Or just run §14 at the end of your last model's session
#  — both work, since it globs Drive.
# ──────────────────────────────────────────────────────────────────────────────
#  Aggregates every metrics_summary_<model>.csv into ONE comparison table, grouped
#  by the defined metric hierarchy (rows) × model (columns), and computes the two
#  controlled contrasts of the design:
#     • Δ_quant : Llama-3.1-8B  Q8_0 − Q4_K_M   (Factor B, quantization effect)
#     • Δ_KG    : GraphRAG − RAG-only, per matched model, IF the GraphRAG arm's
#                 metrics_summary is also present on Drive (the separate KG ablation)
#  It also runs McNemar's paired test (exact binomial) on the per-case correctness
#  vectors of each model vs the anchor, since all models see the identical 90 cases.
#  Outputs: benchmark_comparison.csv, mcnemar_vs_base.csv, benchmark_comparison.png
# ==============================================================================

print("── §14  Cross-Model Comparison ──")
import pandas as pd, glob, os
import numpy as np

BASE_MODEL_KEY = "llama31-8b-q4"          # anchor: Factor-A/Factor-B intersection
QUANT_PAIR     = ("llama31-8b-q8", "llama31-8b-q4")   # (high-bit, low-bit) for Δ_quant

# Metric hierarchy (primary → secondary → tertiary). Rows appear in this order.
METRIC_ORDER = [
    "decision_accuracy", "decision_macro_f1", "decision_mcc",           # Tier 1: prediction
    "article_precision", "article_recall", "article_F1",                # Tier 2: citation
    "ROUGE-1", "ROUGE-2", "ROUGE-L", "BLEU", "BERTScore-F1",            # Tier 3: generation
    "rubric_final_score_pct", "rubric_questions_passed",               # strategy quality
]

# ── 14.1  Aggregate all per-model metrics summaries ───────────────────────────
_cands = sorted(set(glob.glob(os.path.join(DRIVE_ROOT, "results", "**",
                              "metrics_summary*.csv"), recursive=True)
                    + glob.glob("/content/metrics_summary*.csv")))
_frames = []
for p in _cands:
    try:
        df = pd.read_csv(p)
        if "model" not in df.columns:                 # e.g. the GraphRAG arm's summary
            df["model"] = df.get("arm", os.path.basename(p))
        for c in ("arch", "quant"):
            if c not in df.columns: df[c] = "n/a"
        _frames.append(df[["model", "arch", "quant", "metric", "value"]])
    except Exception as e:
        print(f"  ⚠️  {p}: {e}")

if not _frames:
    print("  No metrics_summary CSVs found yet. Run the 5 models first.")
else:
    long_df = pd.concat(_frames, ignore_index=True).drop_duplicates(
        subset=["model", "metric"], keep="last")
    _meta = (long_df[["model", "arch", "quant"]].drop_duplicates()
             .set_index("model").to_dict("index"))

    # ── 14.2  Comparison table: metric (rows) × model (cols), CI bounds kept ──
    pivot = long_df.pivot_table(index="metric", columns="model", values="value", aggfunc="last")
    _ordered = [m for m in METRIC_ORDER if m in pivot.index] + \
               [m for m in pivot.index if m not in METRIC_ORDER]
    pivot = pivot.loc[_ordered]

    # Order columns: Factor-A (Q4_K_M) by size, then the Q8 probe, then other arms.
    def _col_key(m):
        q = _meta.get(m, {}).get("quant", "n/a")
        return (0 if q == "Q4_K_M" else 1 if q == "Q8_0" else 2, str(m))
    pivot = pivot[sorted(pivot.columns, key=_col_key)]

    # ── 14.3  Δ_quant (Factor B): Llama-3.1-8B Q8_0 − Q4_K_M ─────────────────
    _hi, _lo = QUANT_PAIR
    if _hi in pivot.columns and _lo in pivot.columns:
        pivot["Δ_quant(Q8−Q4)"] = pivot[_hi] - pivot[_lo]

    # ── 14.4  Δ_KG (separate ablation): GraphRAG − RAG-only, if present ──────
    _kg_cols  = [c for c in pivot.columns if "NO_KG" not in str(c).upper()
                 and "ragonly" not in str(c).lower() and str(c) in _meta
                 and "graphrag" in str(c).lower()]
    if _kg_cols and BASE_MODEL_KEY in pivot.columns:
        pivot[f"Δ_KG(vs {BASE_MODEL_KEY})"] = pivot[_kg_cols[0]] - pivot[BASE_MODEL_KEY]

    print("\n=== Performance Evaluation — Cross-Model Comparison ===")
    print(pivot.round(4).to_string())
    pivot.round(4).to_csv("/content/benchmark_comparison.csv", encoding="utf-8")

    # ── 14.5  McNemar paired test: each model vs the anchor (per-case) ───────
    from scipy.stats import binomtest
    _pc = sorted(set(glob.glob(os.path.join(DRIVE_ROOT, "results", "**",
                               "benchmark_results*.csv"), recursive=True)
                     + glob.glob("/content/benchmark_results*.csv")))
    _corr = {}   # model -> {case_id: bool correct}
    for p in _pc:
        try:
            d = pd.read_csv(p)
            # Derive the model key robustly: prefer an explicit column, else parse the
            # filename (benchmark_results_ragonly_<MODEL_KEY>.csv). The 'model' column
            # was added late, so older per-case CSVs only have 'arm' — parsing the
            # filename keeps already-completed runs usable without re-execution.
            if "model" in d.columns:
                mk = str(d["model"].iloc[0])
            elif "arm" in d.columns:
                mk = str(d["arm"].iloc[0]).replace("RAG_LLM_NO_KG__", "")
            else:
                mk = (os.path.basename(p)
                      .replace("benchmark_results_ragonly_", "")
                      .replace("benchmark_results_", "").replace(".csv", ""))
            _corr[mk] = dict(zip(d["case_id"], d["correct_decision"].astype(bool)))

#            mk = d["model"].iloc[0] if "model" in d.columns else os.path.basename(p)
#            _corr[mk] = dict(zip(d["case_id"], d["correct_decision"].astype(bool)))
        except Exception as e:
            print(f"  ⚠️  {p}: {e}")

    mcnemar_rows = []
    if BASE_MODEL_KEY in _corr:
        base = _corr[BASE_MODEL_KEY]
        for mk, cm in _corr.items():
            if mk == BASE_MODEL_KEY: continue
            shared = set(base) & set(cm)
            b = sum(1 for c in shared if base[c] and not cm[c])   # base✓ other✗
            c_ = sum(1 for c in shared if not base[c] and cm[c])  # base✗ other✓
            n_disc = b + c_
            pval = binomtest(min(b, c_), n_disc, 0.5).pvalue if n_disc > 0 else 1.0
            mcnemar_rows.append({"model": mk, "vs": BASE_MODEL_KEY, "n_shared": len(shared),
                                 "base_only_correct": b, "model_only_correct": c_,
                                 "discordant": n_disc, "mcnemar_p": round(pval, 4),
                                 "sig_0.05": pval < 0.05})
    if mcnemar_rows:
        mcnemar_df = pd.DataFrame(mcnemar_rows)
        print("\n=== McNemar paired test (per-case correctness vs anchor) ===")
        print(mcnemar_df.to_string(index=False))
        mcnemar_df.to_csv("/content/mcnemar_vs_base.csv", index=False, encoding="utf-8")

    # ── 14.6  Heatmap of the comparison table (model columns only) ───────────
    try:
        import matplotlib.pyplot as plt, seaborn as sns
        _model_cols = [c for c in pivot.columns if not str(c).startswith("Δ")]
        fig, ax = plt.subplots(figsize=(1.6*len(_model_cols)+3, 0.5*len(pivot)+2))
        sns.heatmap(pivot[_model_cols].astype(float), annot=True, fmt=".3f",
                    cmap="viridis", ax=ax, cbar_kws={"label": "metric value"})
        ax.set_title("RAG+LLM Legal — Cross-Model Benchmark (60-case gold set)")
        plt.tight_layout(); plt.savefig("/content/benchmark_comparison.png", dpi=150); plt.show()
    except Exception as e:
        print(f"  ⚠️  Heatmap skipped: {e}")

    # ── 14.7  Persist comparison artifacts to Drive ──────────────────────────
    _cmp_out = os.path.join(DRIVE_ROOT, "results", "_comparison")
    os.makedirs(_cmp_out, exist_ok=True)
    for fn in ["benchmark_comparison.csv", "mcnemar_vs_base.csv", "benchmark_comparison.png"]:
        if os.path.exists(f"/content/{fn}"):
            try: __import__("shutil").copy2(f"/content/{fn}", os.path.join(_cmp_out, fn))
            except Exception as e: print(f"  ⚠️  {fn}: {e}")
    print(f"\n✅  Comparison saved to {_cmp_out}")
